In [ ]:
import os
import re
import csv
import time
import string
import requests
import concurrent.futures

from tqdm import tqdm
from bs4 import BeautifulSoup

In [7]:
# Create session for the request
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
})

In [8]:
# Genres we care about
GUTENBERG_CATEGORIES = [
    'Adventure', 'American Literature', 'British Literature', 'French Literature',
    'German Literature', 'Russian Literature', 'Classics of Literature', 'Biographies',
    'Novels', 'Short Stories', 'Romance', 'Science-Fiction & Fantasy',
    'Science-Fiction and Fantasy', 'Crime, Thrillers & Mystery', 'Crime, Thrillers and Mystery',
    'Mythology, Legends & Folklore', 'Mythology, Legends and Folklore', 'Gothic Fiction','Humour',
    'Children & Young Adult Reading','Children and Young Adult Reading','Historical Novels',
    'Historical Fiction'
]

# Convert list to a set for efficient lookups and set operations
GUTENBERG_CATEGORIES_SET = set(GUTENBERG_CATEGORIES)

In [9]:
def get_book_metadata(book_number):
    """
    Fetches metadata for a single book from Project Gutenberg by parsing the bibrec table.
    
    Checks if the book is in English and matches the desired genre criteria.
    Does NOT download the book text.

    Args:
        book_number (int): The eBook number to check.

    Returns:
        dict: A dictionary with book details if it matches, otherwise None.
    """
    url = f"https://www.gutenberg.org/ebooks/{book_number}"
    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()  # Raise an error for bad responses (4xx or 5xx)
    except requests.exceptions.RequestException as e:
        print(f"eBook {book_number}: Failed to retrieve page. Error: {e}")
        return None

    try:
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the main metadata table
        bibrec_table = soup.find('table', class_='bibrec')
        if not bibrec_table:
            print(f"eBook {book_number}: Skipping, no bibrec table found.")
            return None

        # --- Extract info from the table (your original logic) ---
        title = "Title not found"
        author = "Author not found"
        language = "Language not found"
        genres = set()

        for row in bibrec_table.find_all('tr'):
            header = row.find('th')
            data = row.find('td')
            
            if not header or not data:
                continue

            header_text = header.get_text(strip=True)

            if header_text == 'Title':
                title = data.get_text(strip=True)
            elif header_text == 'Author':
                # Get text, strip newlines, and take the first line
                author = data.get_text(strip=True).split('\n')[0].strip()
            elif header_text == 'Language':
                language = data.get_text(strip=True)

        # --- Extract genres ONLY from the "Similar Books" section (your code) ---
        similar_books_section = soup.find("h2", string="Similar Books")
        if similar_books_section:
            # Find the parent div of the similar books section
            similar_books_div = similar_books_section.find_parent("div")
            if similar_books_div:
                # Find all links that start with "In Category:"
                category_links = similar_books_div.select("a")
                for link in category_links:
                    link_text = link.get_text().strip()
                    if link_text.startswith("In Category:"):
                        genre = link_text.replace("In Category:", "").strip()
                        if genre: # Add if genre is not an empty string
                            genres.add(genre) # Adding to a set automatically handles duplicates

        # --- Filter 1: Language must be English ---
        if 'english' not in language.lower():
            #print(f"eBook {book_number}: Skipping, not English (Lang: {language})")
            return None

        # --- Filter 2: Genre Matching ---
        if not genres:
            #print(f"eBook {book_number}: Skipping '{title}', no genres found.")
            return None
        
        if not genres.issubset(GUTENBERG_CATEGORIES_SET):
            non_matching_genres = genres.difference(GUTENBERG_CATEGORIES_SET)
            #print(f"eBook {book_number}: Skipping '{title}', contains non-approved genres: {non_matching_genres}")
            return None

        # --- Find Plain Text URL ---
        # Look for the link that contains "Plain Text (utf-8)"
        plain_text_tag = soup.find('a', string=re.compile(r'Plain Text UTF-8'))
        if not plain_text_tag:
            #print(f"eBook {book_number}: Skipping '{title}', no Plain Text UTF-8 link found.")
            return None
        
        # Construct the full URL
        plain_text_url = f"https://www.gutenberg.org{plain_text_tag['href']}"

        # --- Success ---
        #print(f"eBook {book_number}: FOUND '{title}' by {author}")
        return {
            'book_number': book_number,
            'title': title,
            'author': author,
            'language': language,
            'genres': list(genres),
            'plain_text_url': plain_text_url
        }

    except Exception as e:
        #print(f"eBook {book_number}: Error parsing HTML for '{url}'. Error: {e}")
        return None

In [10]:
def scrape_all_book_metadata(start_id=1, end_id=75000, max_workers=10):
    """
    Loops through all eBook numbers concurrently using a thread pool.

    Args:
        start_id (int): The first eBook number to check.
        end_id (int): The last eBook number to check.
        max_workers (int): The number of concurrent threads to run.

    Returns:
        dict: A dictionary of book metadata, keyed by book_number.
    """
    print(f"--- Starting CONCURRENT Metadata Scraping from {start_id} to {end_id} using {max_workers} workers ---")
    all_books_metadata = {}

    book_ids_to_check = range(start_id, end_id + 1)
    total_books = len(book_ids_to_check)

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results_iterator = executor.map(get_book_metadata, book_ids_to_check)
        
        for metadata in tqdm(results_iterator, total=total_books, desc="Scraping metadata"):
            if metadata:
                book_number = metadata['book_number']
                title = metadata['title']
                author = metadata['author']
                language = metadata['language']
                genres = metadata['genres']
                plain_text_url = metadata['plain_text_url']
                print(f"Book Number: {book_number}, | {title}")

                all_books_metadata[book_number] = {
                    'book_number': book_number,
                    'title': title,
                    'author': author,
                    'language': language,
                    'genres': genres,
                    'plain_text_url': plain_text_url
                }

    print(f"\n--- Scraping Complete ---")
    print(f"Found {len(all_books_metadata)} matching books out of {total_books} checked.")
    return all_books_metadata

In [11]:
matched_books = scrape_all_book_metadata(start_id=1, end_id=75000)

--- Starting CONCURRENT Metadata Scraping from 1 to 75000 using 10 workers ---


Scraping metadata:   0%|          | 32/75000 [00:02<55:16, 22.61it/s]  

Book Number: 11, | Alice's Adventures in Wonderland
Book Number: 12, | Through the Looking-Glass
Book Number: 15, | Moby-Dick; or, The Whale
Book Number: 16, | Peter Pan :  [Peter and Wendy]
Book Number: 21, | Three hundred Aesop’s fablesTranslated by George Fyler Townsend
Book Number: 24, | O Pioneers!
Book Number: 27, | Far from the Madding Crowd
Book Number: 28, | The Fables of AesopSelected, Told Anew, and Their History Traced
Book Number: 33, | The Scarlet Letter
Book Number: 35, | The Time Machine
Book Number: 36, | The War of the Worlds


Scraping metadata:   0%|          | 50/75000 [00:02<43:36, 28.64it/s]

Book Number: 41, | The Legend of Sleepy Hollow
Book Number: 42, | The Strange Case of Dr. Jekyll and Mr. Hyde
Book Number: 43, | The Strange Case of Dr. Jekyll and Mr. Hyde
Book Number: 44, | The Song of the Lark
Book Number: 45, | Anne of Green Gables
Book Number: 46, | A Christmas Carol in Prose; Being a Ghost Story of Christmas
Book Number: 47, | Anne of Avonlea
Book Number: 51, | Anne of the Island
Book Number: 54, | The Marvelous Land of Oz
Book Number: 55, | The Wonderful Wizard of Oz


...

Book Number: 74980, | The best stories of Sarah Orne Jewett, Volume 2 (of 2)


Scraping metadata: 100%|█████████▉| 74996/75000 [1:38:39<00:00, 16.05it/s]

Book Number: 74993, | Folks from Dixie


Scraping metadata: 100%|██████████| 75000/75000 [1:38:39<00:00, 12.67it/s]

Book Number: 75000, | Folk tales from Tibet :  With illustrations by a Tibetan artist and some verses from Tibetan love-songs

--- Scraping Complete ---
Found 24278 matching books out of 75000 checked.


In [12]:
# --- Function 3: Save Metadata to CSV ---
def save_metadata_to_csv(all_books_metadata, filename='all_books_metadata.csv'):
    """
    Saves the collected book metadata to a CSV file.

    Args:
        all_books_metadata (dict): The dictionary of book metadata.
        filename (str): The name of the CSV file to save.
    """
    if not all_books_metadata:
        print("No metadata to save.")
        return

    # Get the headers from the first item
    # We use .values() because the dict is keyed by book_number
    first_book = next(iter(all_books_metadata.values()))
    headers = first_book.keys()

    print(f"\n--- Saving metadata to {filename} ---")
    
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()
            for book_data in all_books_metadata.values():
                writer.writerow(book_data)
        print(f"Successfully saved {len(all_books_metadata)} records.")
    except IOError as e:
        print(f"Error saving CSV file: {e}")
    except Exception as e:
        print(f"An unexpected error occurred during CSV save: {e}")

In [14]:
save_metadata_to_csv(matched_books, 'gutenberg_metadata.csv')


--- Saving metadata to gutenberg_metadata.csv ---
Successfully saved 24278 records.
